## IE Datathon March 2026 — Intelligent Electric Mobility (Iberdrola)


### Road Routes | Ministry of Transport and Sustainable Mobility
### Source: https://www.transportes.gob.es/ministerio/proyectos-singulares/estudios-de-movilidad-con-big-data/opendata-movilidad?utm_source=chatgpt.com

##### Description: This dataset provides the foundational geographical network. Teams should use it to map the primary interurban arteries, understand traffic flows, and determine the structural backbone of their proposed charging network. It is not just about lines on a map; it is about understanding where the highest volume of long-distance transit occurs.

In [2]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import glob

ROAD_ROUTES_DIR = Path('data/road_routes')

# ── 1. OD Routes (origin → destination via road route, with trip count) ──────
od_files = sorted(glob.glob(str(ROAD_ROUTES_DIR / 'od_rutas' / '*.csv')))
if od_files:
    df_od_routes = pd.concat(
        [pd.read_csv(f, parse_dates=['date']) for f in od_files],
        ignore_index=True
    )
    print(f'✅  OD Routes:  {len(df_od_routes):,} rows × {df_od_routes.shape[1]} cols  ({len(od_files)} date files)')
    print(f'    Dates: {sorted(df_od_routes["date"].dt.date.unique())}')
    display(df_od_routes.head(3))
else:
    df_od_routes = None
    print('⚠️  No OD routes CSVs found — run data/road_routes/download_pipeline.py first')

# ── 2. Segment Info (trip counts per road segment, by distance/GAU/province) ─
seg_files = sorted(glob.glob(str(ROAD_ROUTES_DIR / 'informacion_tramo' / '*.csv')))
if seg_files:
    df_segments = pd.concat(
        [pd.read_csv(f).assign(date=Path(f).stem[:8]) for f in seg_files],
        ignore_index=True
    )
    df_segments['date'] = pd.to_datetime(df_segments['date'], format='%Y%m%d')
    print(f'\n✅  Segment Info: {len(df_segments):,} rows × {df_segments.shape[1]} cols  ({len(seg_files)} date files)')
    display(df_segments.head(3))
else:
    df_segments = None
    print('⚠️  No segment info CSVs found')

# ── 3. Road Segment Geometry (Shapefile) ────────────────────────────────────
shp_path = ROAD_ROUTES_DIR / 'geometria' / 'Geometria_tramos.shp'
if shp_path.exists():
    gdf_tramos = gpd.read_file(shp_path)
    if gdf_tramos.crs and gdf_tramos.crs.to_epsg() != 4326:
        gdf_tramos = gdf_tramos.to_crs(epsg=4326)
    print(f'\n✅  Geometry:    {len(gdf_tramos):,} segments  |  CRS: EPSG:4326')
    print(f'    Columns: {list(gdf_tramos.columns)}')
    display(gdf_tramos.head(3))
else:
    gdf_tramos = None
    print('⚠️  Shapefile not found — run download_pipeline.py to fetch geometry')

# ── 4. Quick summary ─────────────────────────────────────────────────────────
print('\n─── Road Routes Data Summary ───────────────────────────────')
if df_od_routes is not None:
    print(f'  Total trips recorded:    {df_od_routes["trips"].sum():,.0f}')
    print(f'  Unique routes (IGN IDs): {df_od_routes["route_id"].nunique():,}')
    print(f'  Unique OD zone pairs:    {df_od_routes[["origin_zone","destination_zone"]].drop_duplicates().shape[0]:,}')
    print(f'  Avg distance per trip:   {df_od_routes["distance_km"].mean():.1f} km')
if df_segments is not None:
    print(f'  Unique road segments:    {df_segments["segment_id"].nunique():,}')
    print(f'  Avg trips/segment/day:   {df_segments["trips_total"].mean():,.0f}')
if gdf_tramos is not None:
    print(f'  Geometry segments:       {len(gdf_tramos):,}')

✅  OD Routes:  3,648,985 rows × 6 cols  (1 date files)
    Dates: [datetime.date(2024, 3, 31)]


,date,origin_zone,destination_zone,route_id,distance_km,trips
0,2024-03-31,01001,01001,IGN_00001783,0.88,80.65
1,2024-03-31,01001,01001,IGN_00921007,8.57,4.06
2,2024-03-31,01001,01004_AM,IGN_00905950,66.52,2.50



✅  Segment Info: 410,348 rows × 16 cols  (1 date files)


,segment_id,trips_total,main_origin,main_destination,trips_short,trips_medium,trips_long,trips_intra_gau,trips_inter_gau,trips_intra_provincial,trips_inter_provincial,trips_intra_ccaa,trips_inter_ccaa,trips_national,trips_foreign,date
0,100010212376,1199.72,10184_AM,10184_AM,861.31,206.04,132.37,204.56,995.16,859.33,340.39,885.53,314.19,1196.84,2.88,2024-03-31
1,100010215883,1199.72,10184_AM,10184_AM,861.31,206.04,132.37,204.56,995.16,859.33,340.39,885.53,314.19,1196.84,2.88,2024-03-31
2,100010216791,27644.83,10148,10096_AM,1137.25,5029.62,21477.96,15.44,27629.39,1475.12,26169.71,1576.09,26068.74,26611.83,1033.00,2024-03-31


⚠️  Shapefile not found — run download_pipeline.py to fetch geometry

─── Road Routes Data Summary ───────────────────────────────
  Total trips recorded:    31,872,528
  Unique routes (IGN IDs): 2,680,711
  Unique OD zone pairs:    743,451
  Avg distance per trip:   73.3 km
  Unique road segments:    410,348
  Avg trips/segment/day:   8,186


## Electric vehicle charging points - Datasets | National Access Point for Traffic and Mobility:
### Source: https://nap.dgt.es/dataset/puntos-de-recarga-electrica-para-vehiculos

#### Description: Mobility: Knowing where chargers already exist is crucial to avoid redundancies and identify current coverage gaps. Teams must establish this baseline infrastructure before proposing new additions, ensuring their optimization model only suggests new stations where there is a demonstrable deficit.

In [3]:
import requests
from lxml import etree
import pandas as pd

XML_URL = 'https://infocar.dgt.es/datex2/v3/miterd/EnergyInfrastructureTablePublication/electrolineras.xml'

print('Fetching EV charging points XML from DGT...')
r = requests.get(XML_URL, timeout=120)
r.raise_for_status()
print(f'✅  Downloaded {len(r.content)/1024/1024:.1f} MB')

root = etree.fromstring(r.content)

def ftext(el, name):
    res = el.xpath(f'.//*[local-name()="{name}"]/text()')
    return res[0].strip() if res else None

sites = root.xpath('.//*[local-name()="energyInfrastructureSite"]')
print(f'Parsing {len(sites):,} charging sites...')

records = []
for site in sites:
    lat = ftext(site, 'latitude')
    lon = ftext(site, 'longitude')
    if not lat or not lon:
        continue

    name_vals = site.xpath('.//*[local-name()="name"]//*[local-name()="value"]/text()')
    connectors = site.xpath('.//*[local-name()="energyInfrastructureConnector"]')

    powers, ctypes = [], []
    for c in connectors:
        p = ftext(c, 'maxPowerOutput')
        if p:
            try: powers.append(float(p))
            except: pass
        ct = ftext(c, 'connectorType')
        if ct: ctypes.append(ct)

    records.append({
        'site_id':         site.get('id', ''),
        'name':            name_vals[0].strip() if name_vals else '',
        'latitude':        float(lat),
        'longitude':       float(lon),
        'n_connectors':    len(connectors),
        'max_power_kw':    max(powers) / 1000 if powers else None,
        'connector_types': ', '.join(sorted(set(ctypes))) if ctypes else None,
        'operator':        ftext(site, 'operatorName'),
    })

df_chargers = pd.DataFrame(records)

# Basic cleanup — keep only points inside Spain's bounding box
df_chargers = df_chargers[
    df_chargers['latitude'].between(35.0, 44.5) &
    df_chargers['longitude'].between(-9.5, 5.0)
].reset_index(drop=True)

print(f'\n✅  {len(df_chargers):,} charging sites in Spain')
print(f'   Connectors per site (avg): {df_chargers["n_connectors"].mean():.1f}')
print(f'   Max power available:       {df_chargers["max_power_kw"].max()} kW')
display(df_chargers.head(5))

Fetching EV charging points XML from DGT...
✅  Downloaded 79.5 MB
Parsing 12,075 charging sites...

✅  11,567 charging sites in Spain
   Connectors per site (avg): 0.0
   Max power available:       nan kW


,site_id,name,latitude,longitude,n_connectors,max_power_kw,connector_types,operator
0,OPMCKAGOAIX9NOFSBUXT,QWELLO - Calle Juan Antonio Zenón 90,40.288380,-4.020607,0,None,None,None
1,JN6XIVNDB9NVK1KIDMSB,Petrem Eco Moli,42.266670,2.973634,0,None,None,None
2,VEMOQVJHAMQ6RNLG2KAU,PETRO UVE LAVADERO,36.911285,-6.083734,0,None,None,None
3,6I93NNS0EZPXLBAPEMIX,PARKING CENTRO,37.258450,-6.957360,0,None,None,None
4,C2JBB8PHHIZSBOSJMKGB,ESTACION SERVICIO MIRALBUENO S.L,41.663418,-0.930423,0,None,None,None


## Additional Data Sources:

### Iberdrola - Historical Map of Electricity Consumption Capacity
### Source: https://www.i-de.es/conexion-red-electrica/suministro-electrico/mapa-capacidad-consumo

#### Description: Essential for identifying the consumption capacity constraints of the grid in regions managed by Iberdrola. The downloadable dataset provides substation-level data including available capacity (MW) and geographic coordinates.

In [5]:
import pandas as pd
import geopandas as gpd
from pyproj import Transformer
from pathlib import Path

IBERDROLA_PATH = Path('data/Iberdrola Historical Map of Electricity Consumption Capacity/2026_03_05_R1-001_Demanda.csv')

# ── Read raw CSV ─────────────────────────────────────────────────────────────
df_iberdrola = pd.read_csv(
    IBERDROLA_PATH,
    sep=';',
    encoding='utf-8-sig',       # handles the BOM
    decimal=',',                 # Spanish decimal separator
    thousands='.',
    on_bad_lines='skip',
)

# ── Rename columns to English ────────────────────────────────────────────────
df_iberdrola.columns = df_iberdrola.columns.str.strip()
rename_map = {
    'Gestor de red':                                        'grid_operator',
    'Provincia':                                            'province',
    'Municipio':                                            'municipality',
    'Coordenada UTM X':                                     'utm_x',
    'Coordenada UTM Y':                                     'utm_y',
    'Subestación':                                          'substation_id',
    'Nivel de Tensión (kV)':                                'voltage_kv',
    'Capacidad firme disponible (MW)':                      'capacity_available_mw',
    'Capacidad comprometida por cuestiones regulatorias':   'capacity_regulatory_mw',
    'Capacidad de acceso firme de demanda ocupada (MW)':    'capacity_occupied_mw',
    'Capacidad de acceso firme admitida y no evaluada (MW)':'capacity_pending_mw',
}
df_iberdrola = df_iberdrola.rename(columns={k: v for k, v in rename_map.items() if k in df_iberdrola.columns})

# ── Convert UTM (EPSG:25830 — Spain) → WGS84 lat/lon ────────────────────────
transformer = Transformer.from_crs('EPSG:25830', 'EPSG:4326', always_xy=True)
valid_coords = df_iberdrola['utm_x'].notna() & df_iberdrola['utm_y'].notna()
lons, lats = transformer.transform(
    df_iberdrola.loc[valid_coords, 'utm_x'].values,
    df_iberdrola.loc[valid_coords, 'utm_y'].values
)
df_iberdrola.loc[valid_coords, 'longitude'] = lons
df_iberdrola.loc[valid_coords, 'latitude']  = lats

# Keep only points inside Spain
df_iberdrola = df_iberdrola[
    df_iberdrola['latitude'].between(35.0, 44.5) &
    df_iberdrola['longitude'].between(-9.5, 5.0)
].reset_index(drop=True)

# ── GeoDataFrame for spatial operations ─────────────────────────────────────
gdf_iberdrola = gpd.GeoDataFrame(
    df_iberdrola,
    geometry=gpd.points_from_xy(df_iberdrola['longitude'], df_iberdrola['latitude']),
    crs='EPSG:4326'
)

# ── Summary ──────────────────────────────────────────────────────────────────
print(f'✅  {len(df_iberdrola):,} substation records loaded')
print(f'   Provinces covered:         {df_iberdrola["province"].nunique()}')
print(f'   Voltage levels (kV):       {sorted(df_iberdrola["voltage_kv"].dropna().unique().tolist())}')
print(f'   Total available capacity:  {df_iberdrola["capacity_available_mw"].sum():,.1f} MW')
print(f'   Total occupied capacity:   {df_iberdrola["capacity_occupied_mw"].sum():,.1f} MW')
display(df_iberdrola[['grid_operator','province','municipality','voltage_kv',
                       'capacity_available_mw','capacity_occupied_mw','latitude','longitude']].head(8))

✅  3,016 substation records loaded
   Provinces covered:         31
   Voltage levels (kV):       [11.0, 13.2, 15.0, 20.0, 30.0, 45.0, 66.0, 132.0]
   Total available capacity:  2,984.1 MW
   Total occupied capacity:   42,062.0 MW


,grid_operator,province,municipality,voltage_kv,capacity_available_mw,capacity_occupied_mw,latitude,longitude
0,R1-001,Araba/Álava,Ayala/Aiara,30.0,0.0,60.19,43.083669,-3.011056
1,R1-001,Araba/Álava,Vitoria-Gasteiz,13.2,0.0,15.00,42.856076,-2.707190
2,R1-001,Araba/Álava,Vitoria-Gasteiz,30.0,0.0,72.62,42.856076,-2.707190
3,R1-001,Araba/Álava,Vitoria-Gasteiz,13.2,0.0,10.45,42.856076,-2.707190
4,R1-001,Araba/Álava,Vitoria-Gasteiz,30.0,0.0,72.27,42.856076,-2.707190
5,R1-001,Araba/Álava,Vitoria-Gasteiz,30.0,0.0,52.58,42.856076,-2.707190
6,R1-001,Araba/Álava,Vitoria-Gasteiz,30.0,0.0,0.00,42.856076,-2.707190
7,R1-001,Araba/Álava,Lantarón,132.0,0.0,70.00,42.756297,-3.048053


### ENDESA - e-distribucion - Generation access capabilities on network nodes
### Source: https://www.edistribucion.com/es/red-electrica/nodos-capacidad-red/capacidad-generacion.html

#### Description: e-distribución (Network capacity nodes): Provides node-level access capacity, allowing teams to assess the viability of connecting new energy-intensive infrastructure in Endesa's distribution areas. Historical access capacity documents are available for download in CSV and XLSX format.

In [6]:
import pandas as pd
import geopandas as gpd
from pyproj import Transformer
from pathlib import Path

ENDESA_DIR = Path('data/ENDESA - e-distribucion')

# ── Read both files and tag the source ──────────────────────────────────────
dfs = []
for fpath in sorted(ENDESA_DIR.glob('*.csv')):
    df = pd.read_csv(
        fpath,
        sep=';',
        encoding='utf-8-sig',
        decimal=',',
        thousands='.',
        on_bad_lines='skip',
    )
    df['source_file'] = fpath.name
    dfs.append(df)

df_endesa = pd.concat(dfs, ignore_index=True)
df_endesa.columns = df_endesa.columns.str.strip()

# ── Rename columns to English ─────────────────────────────────────────────
rename_map = {
    'Gestor de red':                          'grid_operator',
    'Provincia':                              'province_code',
    'Municipio':                              'municipality_code',
    'Coordenada UTM X':                       'utm_x',
    'Coordenada UTM Y':                       'utm_y',
    'Subestación':                            'substation_id',
    'Nivel de Tensión (kV)':                  'voltage_kv',
    'Capacidad disponible (MW)':              'capacity_available_mw',
    'Capacidad comprometida por cuestiones regulatorias': 'capacity_regulatory_mw',
    'Capacidad ocupada (MW)':                 'capacity_occupied_mw',
    'Capacidad admitida y no resuelta (MW)':  'capacity_pending_mw',
    'Posiciones ocupadas':                    'positions_occupied',
    'Posiciones libres':                      'positions_free',
    'Nudo Afección RdT':                      'node_rdt_affected',
    'Nudo limitado por Scc ':                 'node_scc_limited',
    'Nudo 0*':                                'node_zero',
    'Comentarios':                            'comments',
    'Comunidad Autónoma':                     'autonomous_community',
    'Nombre Subestación':                     'substation_name',
}

# The file has two duplicate cols named Provincia/Municipio — drop the last pair
# (they appear to be substation-level name/province repeats)
cols = df_endesa.columns.tolist()
seen = {}
deduped = []
for c in cols:
    if c in seen:
        seen[c] += 1
        deduped.append(f'{c}_{seen[c]}')
    else:
        seen[c] = 0
        deduped.append(c)
df_endesa.columns = deduped

rename_map['Provincia_1'] = 'province_name'
rename_map['Municipio_1'] = 'municipality_name'

df_endesa = df_endesa.rename(columns={k: v for k, v in rename_map.items() if k in df_endesa.columns})

# ── Convert UTM → WGS84 lat/lon ──────────────────────────────────────────────
transformer = Transformer.from_crs('EPSG:25830', 'EPSG:4326', always_xy=True)
valid = df_endesa['utm_x'].notna() & df_endesa['utm_y'].notna()
lons, lats = transformer.transform(
    df_endesa.loc[valid, 'utm_x'].values,
    df_endesa.loc[valid, 'utm_y'].values
)
df_endesa.loc[valid, 'longitude'] = lons
df_endesa.loc[valid, 'latitude']  = lats

# ── GeoDataFrame ─────────────────────────────────────────────────────────────
gdf_endesa = gpd.GeoDataFrame(
    df_endesa,
    geometry=gpd.points_from_xy(df_endesa['longitude'], df_endesa['latitude']),
    crs='EPSG:4326'
)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f'✅  {len(df_endesa):,} substation records  ({len(dfs)} files)')
print(f'   Grid operators:           {df_endesa["grid_operator"].unique().tolist()}')
print(f'   Voltage levels (kV):      {sorted(df_endesa["voltage_kv"].dropna().unique().tolist())}')
print(f'   Total available capacity: {df_endesa["capacity_available_mw"].sum():,.2f} MW')
print(f'   Total occupied capacity:  {df_endesa["capacity_occupied_mw"].sum():,.2f} MW')
display(df_endesa[['grid_operator', 'substation_name', 'voltage_kv',
                    'capacity_available_mw', 'capacity_occupied_mw',
                    'autonomous_community', 'latitude', 'longitude']].head(10))

✅  1,844 substation records  (2 files)
   Grid operators:           ['R1-026', 'R1-299']
   Voltage levels (kV):      [3, 10, 11, 13, 15, 17, 20, 25, 30, 33, 45, 50, 66, 110, 132]
   Total available capacity: 22,468.50 MW
   Total occupied capacity:  28,433.01 MW


,grid_operator,substation_name,voltage_kv,capacity_available_mw,capacity_occupied_mw,autonomous_community,latitude,longitude
0,R1-026,BAÑOS,3,0.00,0.00,02 - Aragón,42.762126,-0.232417
1,R1-026,ERISTE,25,23.17,0.22,02 - Aragón,42.588931,0.492071
2,R1-026,PUEYO,11,0.00,6.27,02 - Aragón,42.728720,-0.305648
3,R1-026,SABIÑANIGO,11,0.00,25.29,02 - Aragón,42.516817,-0.358014
4,R1-026,SALLENT,20,0.00,2.77,02 - Aragón,42.770602,-0.333731
5,R1-299,AGUADULC,66,0.00,0.00,01 - Andalucía,36.813531,-2.598941
6,R1-299,AGUADULC,20,0.00,0.59,01 - Andalucía,36.813531,-2.598941
7,R1-299,ALBOX,66,0.00,0.00,01 - Andalucía,37.367600,-2.142758
8,R1-299,ALBOX,25,0.00,14.95,01 - Andalucía,37.367600,-2.142758
9,R1-299,ALCOLEA,66,0.00,0.00,01 - Andalucía,36.969459,-2.963403


In [15]:
!pip install pdfplumber -q

import pdfplumber
import pandas as pd
from pathlib import Path

PDF_PATH = Path('data/ENDESA - e-distribucion/Detail Specifications RD Access Capacity August 1, 2025.pdf')

COLUMNS = [
    'autonomous_community', 'province', 'municipality', 'substation',
    'latitude', 'longitude', 'voltage_kv',
    'capacity_available_mw',
    'capacity_occupied_total_mw',
    'capacity_occupied_p1', 'capacity_occupied_p2', 'capacity_occupied_p3',
    'capacity_occupied_p4', 'capacity_occupied_p5', 'capacity_occupied_p6',
    'capacity_occupied_p7', 'capacity_occupied_p8', 'capacity_occupied_p9',
    'capacity_occupied_with_permit', 'capacity_occupied_in_process',
    'capacity_pending_total_mw',
    'capacity_pending_wind', 'capacity_pending_solar_pv',
    'capacity_pending_hydro', 'capacity_pending_solar_thermal',
    'capacity_pending_other',
    'node_scc_limited', 'node_transport_network_affected', 'comments',
]

all_rows = []
with pdfplumber.open(PDF_PATH) as pdf:
    print(f'Extracting tables from {len(pdf.pages)} pages...')
    for page in pdf.pages[1:]:
        table = page.extract_table()
        if not table:
            continue
        for row in table[4:]:
            if row and any(cell for cell in row):
                row = (row + [None] * len(COLUMNS))[:len(COLUMNS)]
                all_rows.append(row)

df_endesa_pdf = pd.DataFrame(all_rows, columns=COLUMNS)

# Clean string columns
str_cols = ['autonomous_community', 'province', 'municipality', 'substation', 'comments']
for col in str_cols:
    df_endesa_pdf[col] = df_endesa_pdf[col].astype(str).str.replace('\n', ' ').str.strip().replace('None', None)

# Clean numeric columns (comma decimal separator)
numeric_cols = [c for c in COLUMNS if c not in str_cols + ['node_scc_limited', 'node_transport_network_affected']]
for col in numeric_cols:
    df_endesa_pdf[col] = pd.to_numeric(
        df_endesa_pdf[col].astype(str).str.replace(',', '.').str.strip().replace({'None': None, '': None, '-': None}),
        errors='coerce'
    )

df_endesa_pdf = df_endesa_pdf.dropna(subset=['substation', 'capacity_available_mw'], how='all').reset_index(drop=True)

print(f'✅  {len(df_endesa_pdf):,} substation records extracted from PDF')
print(f'   Autonomous communities: {df_endesa_pdf["autonomous_community"].nunique()}')
print(f'   Voltage levels (kV):    {sorted(df_endesa_pdf["voltage_kv"].dropna().unique().tolist())}')
print(f'   Total available (MW):   {df_endesa_pdf["capacity_available_mw"].sum():,.2f}')
print(f'   Total occupied (MW):    {df_endesa_pdf["capacity_occupied_total_mw"].sum():,.2f}')
display(df_endesa_pdf.head(10))


Extracting tables from 28 pages...
✅  1,838 substation records extracted from PDF
   Autonomous communities: 10
   Voltage levels (kV):    [3.0, 10.0, 11.0, 13.0, 15.0, 17.0, 20.0, 25.0, 30.0, 33.0, 45.0, 50.0, 66.0, 110.0, 132.0]
   Total available (MW):   20,919.60
   Total occupied (MW):    24,654.00


,autonomous_community,province,municipality,substation,latitude,longitude,voltage_kv,capacity_available_mw,capacity_occupied_total_mw,capacity_occupied_p1,...,capacity_occupied_in_process,capacity_pending_total_mw,capacity_pending_wind,capacity_pending_solar_pv,capacity_pending_hydro,capacity_pending_solar_thermal,capacity_pending_other,node_scc_limited,node_transport_network_affected,comments
0,01 - Andalucía,Almería,Roquetas de Mar,AGUADULC,36.813531,-2.598941,66.0,0.0,0.0,NaN,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NO,BERJA 220,
1,01 - Andalucía,Almería,Roquetas de Mar,AGUADULC,36.813531,-2.598941,20.0,0.0,0.0,NaN,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NO,BERJA 220,
2,01 - Andalucía,Almería,Albox,ALBOX,37.367600,-2.142758,66.0,0.0,0.0,NaN,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NO,LITORAL 400,
3,01 - Andalucía,Almería,Albox,ALBOX,37.367600,-2.142758,25.0,0.0,0.0,NaN,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NO,LITORAL 400,
4,01 - Andalucía,Almería,Alcolea,ALCOLEA,36.969459,-2.963403,66.0,0.0,0.0,NaN,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NO,BERJA 220,
5,01 - Andalucía,Almería,Alcolea,ALCOLEA,36.969459,-2.963403,20.0,0.0,0.0,NaN,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NO,BERJA 220,
6,01 - Andalucía,Almería,Almería,ANDARAX,36.839727,-2.431844,132.0,0.0,0.0,NaN,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NO,LITORAL 400,
7,01 - Andalucía,Almería,Almería,ANDARAX,36.839727,-2.431844,66.0,0.0,0.0,NaN,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NO,LITORAL 400,
8,01 - Andalucía,Almería,Almería,ANDARAX,36.839727,-2.431844,20.0,0.0,0.0,NaN,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NO,LITORAL 400,
9,01 - Andalucía,Almería,Almería,BELEN,36.850365,-2.474578,66.0,0.0,0.0,NaN,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NO,LITORAL 400,


### VIESGO: Interactive grid map: Provides substation-level capacity data for Viesgo's distribution network in northern Spain, critical for cross-referencing geographical charging needs with electrical supply capabilities
### Source: https://www.viesgodistribucion.com/mapa-interactivo-de-la-red

#### Description: Provides substation-level capacity data for Viesgo's distribution network in northern Spain, critical for cross-referencing geographical charging needs with electrical supply capabilities

In [17]:
import pandas as pd
import geopandas as gpd
from pyproj import Transformer
from pathlib import Path

VIESGO_DIR = Path('data/VIESGO')

df_viesgo_demand = pd.read_csv(
    VIESGO_DIR / '2026_04_01_R1005_demanda_Consumption capacity in the network.csv',
    sep=';',
    encoding='latin-1',
    skip_blank_lines=True,
)
df_viesgo_demand.columns = df_viesgo_demand.columns.str.strip()

rename_map = {
    'Gestor de red':                                         'grid_operator',
    'Provincia':                                             'province',
    'Municipio':                                             'municipality',
    'Coordenada UTM X':                                      'utm_x_raw',
    'Coordenada UTM Y':                                      'utm_y_raw',
    'Subestación ':                                          'substation_id',
    'Subestación':                                           'substation_id',
    'Nivel de tensión (kV)':                                 'voltage_kv',
    'Capacidad firme disponible (MW)':                       'capacity_available_mw',
    'Capacidad comprometida por cuestiones regulatorias':    'capacity_regulatory_mw',
    'Capacidad de acceso firme de demanda ocupada (MW)':     'capacity_occupied_mw',
    'Capacidad de acceso firme admitida y no evaluada (MW)': 'capacity_pending_mw',
    'Posiciones ocupadas':                                   'positions_occupied',
    'Posiciones libres':                                     'positions_free',
    'Nudo 0*':                                               'node_zero',
    'Comentarios':                                           'comments',
    'Nombre subestación':                                    'substation_name',
}
df_viesgo_demand = df_viesgo_demand.rename(columns={k: v for k, v in rename_map.items() if k in df_viesgo_demand.columns})

def parse_spanish_utm(series):
    return pd.to_numeric(
        series.astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False),
        errors='coerce'
    )

df_viesgo_demand['utm_x'] = parse_spanish_utm(df_viesgo_demand['utm_x_raw'])
df_viesgo_demand['utm_y'] = parse_spanish_utm(df_viesgo_demand['utm_y_raw'])

for col in ['voltage_kv', 'capacity_available_mw', 'capacity_regulatory_mw',
            'capacity_occupied_mw', 'capacity_pending_mw']:
    if col in df_viesgo_demand.columns:
        df_viesgo_demand[col] = pd.to_numeric(
            df_viesgo_demand[col].astype(str).str.replace(',', '.').str.strip(),
            errors='coerce'
        )

transformer = Transformer.from_crs('EPSG:25830', 'EPSG:4326', always_xy=True)
valid = df_viesgo_demand['utm_x'].notna() & df_viesgo_demand['utm_y'].notna()
lons, lats = transformer.transform(df_viesgo_demand.loc[valid, 'utm_x'].values,
                                    df_viesgo_demand.loc[valid, 'utm_y'].values)
df_viesgo_demand.loc[valid, 'longitude'] = lons
df_viesgo_demand.loc[valid, 'latitude']  = lats

df_viesgo_demand = df_viesgo_demand.dropna(subset=['substation_id']).reset_index(drop=True)

gdf_viesgo_demand = gpd.GeoDataFrame(
    df_viesgo_demand,
    geometry=gpd.points_from_xy(df_viesgo_demand['longitude'], df_viesgo_demand['latitude']),
    crs='EPSG:4326'
)

print(f'✅  Viesgo Demand: {len(df_viesgo_demand):,} records')
print(f'   Provinces:            {sorted(df_viesgo_demand["province"].dropna().unique().tolist())}')
print(f'   Voltage levels (kV):  {sorted(df_viesgo_demand["voltage_kv"].dropna().unique().tolist())}')
print(f'   Total available (MW): {df_viesgo_demand["capacity_available_mw"].sum():,.2f}')
print(f'   Total occupied (MW):  {df_viesgo_demand["capacity_occupied_mw"].sum():,.2f}')
display(df_viesgo_demand[['grid_operator','substation_name','province','voltage_kv',
                           'capacity_available_mw','capacity_occupied_mw','latitude','longitude']].head(8))


✅  Viesgo Demand: 177 records
   Provinces:            ['Asturias', 'Burgos', 'Cantabria', 'Lugo', 'Palencia', 'Tarragona']
   Voltage levels (kV):  [12, 20, 25, 30, 55, 132]
   Total available (MW): 1,042.79
   Total occupied (MW):  1,675.22


,grid_operator,substation_name,province,voltage_kv,capacity_available_mw,capacity_occupied_mw,latitude,longitude
0,R1-005,ALMUÑA,Asturias,20,8.97,16.03,43.531231,-6.522581
1,R1-005,ALMUÑA,Asturias,132,30.00,0.00,43.531231,-6.522581
2,R1-005,ARBON,Asturias,30,1.91,23.09,43.478805,-6.730785
3,R1-005,ARBON,Asturias,132,30.00,79.00,43.478805,-6.730785
4,R1-005,ARENAS,Asturias,132,3.00,0.00,43.300845,-4.809832
5,R1-005,BAIÑA,Asturias,12,0.00,1.95,43.274761,-5.827762
6,R1-005,BAIÑA,Asturias,30,1.39,0.60,43.274761,-5.827762
7,R1-005,CAMARMEÑA,Asturias,132,3.00,0.00,43.260082,-4.831408


In [19]:
import pandas as pd
import geopandas as gpd
from pyproj import Transformer
from pathlib import Path

VIESGO_DIR = Path('data/VIESGO')

df_viesgo_gen = pd.read_csv(
    VIESGO_DIR / '2026_04_01_R1005_generacion_Generation capacity in the network.csv',
    sep=';',
    encoding='latin-1',
    skip_blank_lines=True,
)
df_viesgo_gen.columns = df_viesgo_gen.columns.str.strip().str.replace('\n', ' ', regex=False)

rename_map = {
    'Gestor de red':                           'grid_operator',
    'Provincia':                               'province',
    'Municipio':                               'municipality',
    'Coordenada UTM X':                        'utm_x_raw',
    'Coordenada UTM Y':                        'utm_y_raw',
    'Subestación':                             'substation_id',
    'Nivel de Tensión (kV)':                   'voltage_kv',
    'Capacidad disponible (MW)':               'capacity_available_mw',
    'Capacidad comprometida por cuestiones regulatorias': 'capacity_regulatory_mw',
    'Capacidad ocupada (MW)':                  'capacity_occupied_mw',
    'Capacidad admitida y no resuelta (MW)':   'capacity_pending_mw',
    'Posiciones ocupadas':                     'positions_occupied',
    'Posiciones libres':                       'positions_free',
    'Nudo Afección RdT':                       'node_rdt_affected',
    'Nudo limitado por Scc':                   'node_scc_limited',
    'Nudo 0*':                                 'node_zero',
    'Comentarios':                             'comments',
    'Nombre Subestación':                      'substation_name',
}

tech_renames = {}
for col in df_viesgo_gen.columns:
    col_lower = col.lower()
    if 'lica' in col_lower and 'admitida' in col_lower:
        tech_renames[col] = 'capacity_pending_wind_mw'
    elif 'solar' in col_lower and 'admitida' in col_lower:
        tech_renames[col] = 'capacity_pending_solar_mw'
    elif 'almacenamiento' in col_lower:
        tech_renames[col] = 'capacity_pending_storage_mw'
    elif 'otras' in col_lower and 'admitida' in col_lower:
        tech_renames[col] = 'capacity_pending_other_mw'
rename_map.update(tech_renames)

df_viesgo_gen = df_viesgo_gen.rename(columns={k: v for k, v in rename_map.items() if k in df_viesgo_gen.columns})

def parse_spanish_utm(series):
    return pd.to_numeric(
        series.astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False),
        errors='coerce'
    )

df_viesgo_gen['utm_x'] = parse_spanish_utm(df_viesgo_gen['utm_x_raw'])
df_viesgo_gen['utm_y'] = parse_spanish_utm(df_viesgo_gen['utm_y_raw'])

numeric_cols = ['voltage_kv', 'capacity_available_mw', 'capacity_regulatory_mw',
                'capacity_occupied_mw', 'capacity_pending_mw',
                'capacity_pending_wind_mw', 'capacity_pending_solar_mw',
                'capacity_pending_storage_mw', 'capacity_pending_other_mw']
for col in numeric_cols:
    if col in df_viesgo_gen.columns:
        df_viesgo_gen[col] = pd.to_numeric(
            df_viesgo_gen[col].astype(str).str.replace(',', '.').str.strip(),
            errors='coerce'
        )

transformer = Transformer.from_crs('EPSG:25830', 'EPSG:4326', always_xy=True)
valid = df_viesgo_gen['utm_x'].notna() & df_viesgo_gen['utm_y'].notna()
lons, lats = transformer.transform(df_viesgo_gen.loc[valid, 'utm_x'].values,
                                    df_viesgo_gen.loc[valid, 'utm_y'].values)
df_viesgo_gen.loc[valid, 'longitude'] = lons
df_viesgo_gen.loc[valid, 'latitude']  = lats

df_viesgo_gen = df_viesgo_gen.dropna(subset=['substation_id']).reset_index(drop=True)

gdf_viesgo_gen = gpd.GeoDataFrame(
    df_viesgo_gen,
    geometry=gpd.points_from_xy(df_viesgo_gen['longitude'], df_viesgo_gen['latitude']),
    crs='EPSG:4326'
)

print(f'✅  Viesgo Generation: {len(df_viesgo_gen):,} records')
print(f'   Provinces:            {sorted(df_viesgo_gen["province"].dropna().unique().tolist())}')
print(f'   Voltage levels (kV):  {sorted(df_viesgo_gen["voltage_kv"].dropna().unique().tolist())}')
print(f'   Total available (MW): {df_viesgo_gen["capacity_available_mw"].sum():,.2f}')
print(f'   Total occupied (MW):  {df_viesgo_gen["capacity_occupied_mw"].sum():,.2f}')
display(df_viesgo_gen[['grid_operator','substation_name','province','voltage_kv',
                        'capacity_available_mw','capacity_occupied_mw',
                        'capacity_pending_wind_mw','capacity_pending_solar_mw',
                        'latitude','longitude']].head(8))

✅  Viesgo Generation: 177 records
   Provinces:            ['Asturias', 'Burgos', 'Cantabria', 'Lugo', 'Palencia', 'Tarragona']
   Voltage levels (kV):  [12.0, 20.0, 25.0, 30.0, 55.0, 132.0]
   Total available (MW): 741.11
   Total occupied (MW):  3,643.50


,grid_operator,substation_name,province,voltage_kv,capacity_available_mw,capacity_occupied_mw,capacity_pending_wind_mw,capacity_pending_solar_mw,latitude,longitude
0,R1-005,ALMUÑA,Asturias,20.0,0.0,0.000,0.0,0.0,43.531231,-6.522581
1,R1-005,ALMUÑA,Asturias,132.0,0.0,0.000,0.0,0.0,43.531231,-6.522581
2,R1-005,ARBON,Asturias,30.0,0.0,24.000,0.0,0.0,43.478805,-6.730785
3,R1-005,ARBON,Asturias,132.0,0.0,133.015,0.0,0.0,43.478805,-6.730785
4,R1-005,ARENAS,Asturias,132.0,0.0,8.300,0.0,0.0,43.300845,-4.809832
5,R1-005,BAIÑA,Asturias,12.0,0.0,0.000,0.0,0.0,43.274761,-5.827762
6,R1-005,BAIÑA,Asturias,30.0,0.0,0.000,0.0,0.0,43.274761,-5.827762
7,R1-005,CAMARMEÑA,Asturias,132.0,0.0,12.770,0.0,0.0,43.260082,-4.831408


### Vehicle registrations in Spain (DGT)
### Source: https://www.dgt.es/menusecundario/dgt-en-cifras/matraba-listados/matriculaciones-automoviles-mensual.html


#### This provides the historical and current context of the vehicle fleet. By analyzing monthly registration trends, teams can gauge the localized pace of EV adoption across different provinces, allowing them to weight their charging demand forecasts geographically.

In [21]:
import requests, zipfile, io
import pandas as pd
from pathlib import Path

# ── Fixed-width column spec from MATRABA format PDF ──────────────────────────
COLSPECS = [
    (0,   8,  'registration_date'),
    (8,   9,  'registration_class'),
    (9,   17, 'processing_date'),
    (17,  47, 'brand'),
    (47,  69, 'model'),
    (69,  70, 'origin_code'),
    (70,  91, 'chassis'),
    (91,  93, 'vehicle_type_code'),
    (93,  94, 'propulsion_code'),
    (94,  99, 'engine_cc'),
    (99,  105, 'fiscal_power_cvf'),
    (105, 111, 'tare_kg'),
    (111, 117, 'max_weight_kg'),
    (117, 120, 'seats'),
    (120, 122, 'ind_precinto'),
    (122, 124, 'ind_embargo'),
    (124, 126, 'num_transfers'),
    (126, 128, 'num_owners'),
    (128, 152, 'locality'),
    (152, 154, 'province_code_vehicle'),
    (154, 156, 'province_code_reg'),
    (156, 157, 'procedure_code'),
    (157, 165, 'procedure_date'),
    (165, 170, 'postal_code'),
    (170, 178, 'first_reg_date'),
    (178, 179, 'new_used'),
    (179, 180, 'person_type'),
    (180, 189, 'itv_code'),
    (189, 192, 'service_code'),
    (192, 197, 'municipality_ine'),
    (197, 227, 'municipality'),
    (227, 234, 'power_kw'),
    (234, 237, 'seats_max'),
    (237, 242, 'co2_emissions'),
    (242, 243, 'renting'),
    (243, 244, 'tutela_code'),
    (244, 245, 'possession_code'),
    (245, 246, 'baja_definitiva'),
    (246, 247, 'baja_temporal'),
    (247, 248, 'stolen'),
    (248, 259, 'baja_telematica'),
    (259, 284, 'vehicle_type'),
    (284, 309, 'variant'),
    (309, 344, 'version'),
    (344, 414, 'manufacturer'),
    (414, 420, 'mass_running_order'),
    (420, 426, 'mass_max_admissible'),
    (426, 430, 'eu_homologation_cat'),
    (430, 434, 'body_type'),
    (434, 437, 'standing_seats'),
    (437, 445, 'euro_emissions_level'),
    (445, 449, 'consumption_wh_km'),
    (449, 453, 'regulation_class'),
    (453, 457, 'ev_category'),
    (457, 463, 'ev_range_km'),
    (463, 493, 'base_vehicle_brand'),
    (493, 543, 'base_vehicle_maker'),
    (543, 578, 'base_vehicle_type'),
    (578, 603, 'base_vehicle_variant'),
    (603, 638, 'base_vehicle_version'),
    (638, 642, 'axle_distance_12'),
    (642, 646, 'front_track'),
    (646, 650, 'rear_track'),
    (650, 651, 'fuel_type'),
    (651, 676, 'homologation_code'),
    (676, 677, 'eco_innovation'),
    (677, 681, 'eco_reduction'),
    (681, 706, 'eco_code'),
    (706, 714, 'process_date'),
]

PROPULSION_MAP = {
    '1': 'Gasoline', '2': 'Diesel', '3': 'Electric',
    '4': 'Hybrid_gasoline', '5': 'Hybrid_diesel',
    '6': 'LPG', '7': 'CNG', '8': 'Hydrogen',
    'H': 'Hybrid_rechargeable_gasoline',
    'I': 'Hybrid_rechargeable_diesel',
    'K': 'Hybrid_rechargeable_other',
    'L': 'Other_alternative', 'M': 'Multi_fuel',
}

def build_url(year: int, month: int) -> str:
    return (f'https://www.dgt.es/microdatos/salida/{year}/{month}'
            f'/vehiculos/matriculaciones/export_mensual_mat_{year}{month:02d}.zip')

def load_month(year: int, month: int) -> pd.DataFrame:
    url = build_url(year, month)
    r = requests.get(url, timeout=120, headers={'User-Agent': 'Mozilla/5.0'})
    if r.status_code != 200:
        print(f'  ⚠️  {year}-{month:02d}: HTTP {r.status_code}')
        return pd.DataFrame()

    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        fname = z.namelist()[0]
        with z.open(fname) as f:
            content = f.read().decode('latin-1')

    lines = [l for l in content.splitlines() if len(l) >= 200]

    rows = []
    for line in lines:
        row = {name: line[start:end].strip() for start, end, name in COLSPECS}
        rows.append(row)

    df = pd.DataFrame(rows)
    df['year']  = year
    df['month'] = month

    for col in ['registration_date', 'first_reg_date', 'process_date']:
        df[col] = pd.to_datetime(df[col], format='%d%m%Y', errors='coerce')

    for col in ['power_kw', 'co2_emissions', 'ev_range_km', 'consumption_wh_km',
                'engine_cc', 'tare_kg', 'max_weight_kg', 'seats']:
        df[col] = pd.to_numeric(df[col].replace('*', None), errors='coerce')

    df['propulsion'] = df['propulsion_code'].map(PROPULSION_MAP).fillna('Other')

    return df


# ── Load all months from 2020 onwards ────────────────────────────────────────
YEARS_MONTHS = [
    (year, month)
    for year in range(2020, 2027)
    for month in range(1, 13)
    if (year, month) <= (2026, 3)
]

dfs = []
for year, month in YEARS_MONTHS:
    print(f'Loading {year}-{month:02d} ...', end=' ')
    df = load_month(year, month)
    if len(df):
        print(f'{len(df):,} records')
        dfs.append(df)

df_registrations = pd.concat(dfs, ignore_index=True)

# ── EV subset ─────────────────────────────────────────────────────────────────
EV_CODES = ['3', 'H', 'I', 'K']
df_ev_registrations = df_registrations[
    df_registrations['propulsion_code'].isin(EV_CODES)
].copy()

# ── Summary ───────────────────────────────────────────────────────────────────
print(f'\n✅  Total registrations loaded: {len(df_registrations):,}')
print(f'   EV registrations:           {len(df_ev_registrations):,}')
print(f'   Period: {df_registrations["registration_date"].min().date()} → '
      f'{df_registrations["registration_date"].max().date()}')
print(f'\nTop 10 EV brands:')
print(df_ev_registrations['brand'].value_counts().head(10).to_string())
print(f'\nEV registrations by province (top 10):')
print(df_ev_registrations['province_code_vehicle'].value_counts().head(10).to_string())

display(df_ev_registrations[['registration_date','brand','model','propulsion',
                              'power_kw','ev_range_km','province_code_vehicle',
                              'municipality','new_used']].head(10))

Loading 2020-01 ... 133,560 records
Loading 2020-02 ... 144,158 records
Loading 2020-03 ... 64,218 records
Loading 2020-04 ... 10,682 records
Loading 2020-05 ... 61,532 records
Loading 2020-06 ... 140,071 records
Loading 2020-07 ... 186,781 records
Loading 2020-08 ... 111,895 records
Loading 2020-09 ... 121,791 records
Loading 2020-10 ... 128,042 records
Loading 2020-11 ... 123,822 records
Loading 2020-12 ... 156,720 records
Loading 2021-01 ... 75,335 records
Loading 2021-02 ... 100,919 records
Loading 2021-03 ... 140,588 records
Loading 2021-04 ... 129,073 records
Loading 2021-05 ... 148,589 records
Loading 2021-06 ... 153,744 records
Loading 2021-07 ... 138,044 records
Loading 2021-08 ... 85,084 records
Loading 2021-09 ... 106,942 records
Loading 2021-10 ... 107,225 records
Loading 2021-11 ... 115,177 records
Loading 2021-12 ... 133,053 records
Loading 2022-01 ... 80,320 records
Loading 2022-02 ... 105,956 records
Loading 2022-03 ... 109,184 records
Loading 2022-04 ... 113,859 record


KeyboardInterrupt



### Electric car charging points | datos.gob.es (e.g., Municipal Dataset of Vigo)
### Source: https://datos.vigo.org/data/trafico/ptos_recarga.geojson

#### Description: While national registries provide the macro picture, teams are encouraged to search for local open data portals (like the provided example from Vigo, and other municipalities on datos.gob.es). These local datasets can help validate national figures, identify unregistered regional chargers, and provide granular insights to enrich your model.

In [22]:
import requests
import geopandas as gpd
import pandas as pd
import io

URL = 'https://datos.vigo.org/data/trafico/ptos_recarga.geojson'

print('Fetching GeoJSON...')
r = requests.get(URL, timeout=30)
r.raise_for_status()
print(f'✅  {len(r.content)/1024:.1f} KB downloaded')

gdf_vigo_chargers = gpd.read_file(io.BytesIO(r.content))

# Ensure WGS84
if gdf_vigo_chargers.crs and gdf_vigo_chargers.crs.to_epsg() != 4326:
    gdf_vigo_chargers = gdf_vigo_chargers.to_crs(epsg=4326)

# Extract lat/lon from geometry into plain columns
gdf_vigo_chargers['longitude'] = gdf_vigo_chargers.geometry.x
gdf_vigo_chargers['latitude']  = gdf_vigo_chargers.geometry.y

print(f'✅  {len(gdf_vigo_chargers):,} charging points loaded')
print(f'   Columns: {list(gdf_vigo_chargers.columns)}')
print(f'   CRS: {gdf_vigo_chargers.crs}')
display(gdf_vigo_chargers.head(10))

Fetching GeoJSON...
✅  4.3 KB downloaded
✅  13 charging points loaded
   Columns: ['barrio', 'codigo_postal', 'numero', 'web', 'calle', 'parroquia', 'lon', 'id', 'telefono', 'nombre', 'lat', 'geometry', 'longitude', 'latitude']
   CRS: EPSG:4326


,barrio,codigo_postal,numero,web,calle,parroquia,lon,id,telefono,nombre,lat,geometry,longitude,latitude
0,FREIXEIRO,36210,2.0,http://www.granviadevigo.com,RUA MIRADOIRO (DO),FREIXEIRO,-8.72426,5390,986447500,Parking Centro Comercial Gran Vía,42.22039,POINT (-8.72426 42.22039),-8.72426,42.22039
1,VIGO,36201,0.0,http://www.eloymartranvias.com/es/urzaiz.html,RUA URZAIZ,VIGO,-8.71962,5112,986442339,Parking Urzáiz,42.23596,POINT (-8.71962 42.23596),-8.71962,42.23596
2,None,36201,NaN,None,RUA AREAL,None,-8.71860,5127,986223225,Parking Areal,42.23953,POINT (-8.7186 42.23953),-8.71860,42.23953
3,VIGO,36211,NaN,http://www.eloymartranvias.com/es/colmeiro.html,RUA PINTOR COLMEIRO,VIGO,-8.72597,5128,986413419,Parking Pintor Colmeiro,42.22535,POINT (-8.72597 42.22535),-8.72597,42.22535
4,None,None,0.0,http://www.eloymartranvias.com/es/policarpo.html,RUA POLICARPO SANZ,None,-8.72414,5129,986229404,Parking Policarpo Sanz,42.23760,POINT (-8.72414 42.2376),-8.72414,42.23760
5,VIGO,36211,0.0,http://www.eloymartranvias.com/es/independenci...,PRAZA INDEPENDENCIA,VIGO,-8.73014,5130,None,Parking Independencia,42.22373,POINT (-8.73014 42.22373),-8.73014,42.22373
6,None,None,NaN,http://www.empark.com,PRAZA PORTUGAL,None,-8.71847,5389,None,Parking Praza Portugal,42.23596,POINT (-8.71847 42.23596),-8.71847,42.23596
7,VIGO,36207,202.0,https://www.cctravesia.com/,RUA TRAVESIA DE VIGO,VIGO,-8.69659,5391,986277908,Centro Comercial Travesía de Vigo,42.24307,POINT (-8.69659 42.24307),-8.69659,42.24307
8,BABIO,36212,341.0,https://xxivigo.sergas.gal/Paxinas/web.aspx,ESTDA CLARA CAMPOAMOR,BEADE,-8.71528,5392,986811111,Parking interior Hospital Álvaro Cunqueiro,42.18974,POINT (-8.71528 42.18974),-8.71528,42.18974
9,None,None,NaN,http://www.uvigo.gal,RUA CUMIEIRA (DA),None,-8.67669,5393,None,Parking Escola Universitaria Filoloxía,42.16991,POINT (-8.67669 42.16991),-8.67669,42.16991
